# Lesson 1: A bigram language model

Run the code cells from top to bottom with **Shift+Enter**. Each cell keeps its variables available for the next cell. Select a Python kernel with PyTorch installed.

The training cell runs 5,000 steps; lower `training_steps` for a shorter run. Running it again continues training the current model. To start over, restart the kernel and run from the top.

## Imports

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

## 1. Load training text

In [2]:
from pathlib import Path

# Support opening the notebook from this folder or the project root.
input_path = Path("input.txt")
if not input_path.is_file():
    input_path = Path("chatgpt/input.txt")

with input_path.open("r", encoding="utf-8") as f:
    text = f.read()

print("First 200 characters:")
print(text[:200])

print("\nText length:", len(text))

First 200 characters:
hello world
hello machine learning
language models are interesting
hello pytorch

Text length: 80


## 2. Build a character-level tokenizer

In [3]:
chars = sorted(list(set(text)))

vocab_size = len(chars)

print("\nCharacters:")
print(chars)

print("\nVocabulary size:", vocab_size)


# string -> integer
stoi = {
    ch: i
    for i, ch in enumerate(chars)
}

# integer -> string
itos = {
    i: ch
    for i, ch in enumerate(chars)
}


def encode(s):
    return [stoi[c] for c in s]


def decode(ids):
    return "".join(itos[i] for i in ids)


# Test tokenizer
test_text = "hello"

print("\nTokenizer test:")
print("Original:", test_text)
print("Encoded :", encode(test_text))
print("Decoded :", decode(encode(test_text)))


Characters:
['\n', ' ', 'a', 'c', 'd', 'e', 'g', 'h', 'i', 'l', 'm', 'n', 'o', 'p', 'r', 's', 't', 'u', 'w', 'y']

Vocabulary size: 20

Tokenizer test:
Original: hello
Encoded : [7, 5, 9, 9, 12]
Decoded : hello


## 3. Convert the whole dataset to token IDs

In [4]:
data = torch.tensor(
    encode(text),
    dtype=torch.long
)

print("\nFirst 20 token IDs:")
print(data[:20])

print("Data shape:", data.shape)


First 20 token IDs:
tensor([ 7,  5,  9,  9, 12,  1, 18, 12, 14,  9,  4,  0,  7,  5,  9,  9, 12,  1,
        10,  2])
Data shape: torch.Size([80])


## 4. Create input / target pairs

```text
Example:

x = [a, b, c]
y = [b, c, d]

The model learns:

a -> b
b -> c
c -> d
```

In [5]:
x = data[:-1]
y = data[1:]

print("\nFirst 10 input tokens:")
print(x[:10])

print("\nFirst 10 target tokens:")
print(y[:10])


First 10 input tokens:
tensor([ 7,  5,  9,  9, 12,  1, 18, 12, 14,  9])

First 10 target tokens:
tensor([ 5,  9,  9, 12,  1, 18, 12, 14,  9,  4])


## 5. Bigram Language Model

In [6]:
class BigramLanguageModel(nn.Module):

    def __init__(self, vocab_size):
        super().__init__()

        # Each token points to a row containing
        # scores for every possible next token.
        self.token_embedding_table = nn.Embedding(
            vocab_size,
            vocab_size
        )

    def forward(self, idx, targets=None):

        # idx shape:
        # [number_of_tokens]
        #
        # logits shape:
        # [number_of_tokens, vocab_size]

        logits = self.token_embedding_table(idx)

        loss = None

        if targets is not None:

            loss = F.cross_entropy(
                logits,
                targets
            )

        return logits, loss

## 6. Create the model

In [7]:
model = BigramLanguageModel(vocab_size)


# Test model before training
logits, loss = model(x, y)

print("\nInitial loss:")
print(loss.item())


Initial loss:
3.5093510150909424


## 7. Optimizer

In [8]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-2
)

## 8. Training

In [9]:
training_steps = 5000

for step in range(training_steps):

    # Forward pass
    logits, loss = model(x, y)

    # Remove gradients from previous training step
    optimizer.zero_grad()

    # Calculate gradients
    loss.backward()

    # Update weights
    optimizer.step()

    if step % 500 == 0:
        print(
            f"step {step}: "
            f"loss = {loss.item():.4f}"
        )

step 0: loss = 3.5094
step 500: loss = 1.1611
step 1000: loss = 1.1063
step 1500: loss = 1.0949
step 2000: loss = 1.0903
step 2500: loss = 1.0878
step 3000: loss = 1.0863
step 3500: loss = 1.0852
step 4000: loss = 1.0845
step 4500: loss = 1.0839


## 9. Text generation

In [10]:
@torch.no_grad()
def generate(
    model,
    idx,
    max_new_tokens
):

    for _ in range(max_new_tokens):

        # Predict next-token logits
        logits, _ = model(idx)

        # We only care about prediction
        # after the final token
        logits = logits[-1]

        # Convert logits to probabilities
        probs = F.softmax(
            logits,
            dim=-1
        )

        # Randomly sample next token
        next_token = torch.multinomial(
            probs,
            num_samples=1
        )

        # Add next token to sequence
        idx = torch.cat(
            [idx, next_token]
        )

    return idx

## 10. Generate some text

In [11]:
# Start generation with character "h"
start = torch.tensor(
    [encode("h")[0]],
    dtype=torch.long
)

generated = generate(
    model,
    start,
    max_new_tokens=300
)

print("\nGenerated text:")
print(
    decode(
        generated.tolist()
    )
)


Generated text:
herche inining
hintorelllllls morngeanintinguarche achelorchelanting
helarcheres magellde mo into mo aguachere are lderellode lod
helellodelod
herernte achinerle agello morchineachinining
helorestorchinguange pytornge marelore mag
hing
helod
hing
lls marchelo mo pytodeachelllo pyto wo achinteldeage a
